# MultiMNIST — Entropic LMO-MGDA ablations only

This notebook contains the ablations moved out of the comparison notebook. **It never runs a baseline or a hyperparameter search.**

By default, it loads the newly selected `ours_retune_v3` configuration and writes ablations to a separate directory. Finish comparison notebook cell 8 first. For controlled one-factor ablations, each variant keeps that selected model LR, schedule, eta and momentum injection except for the setting being varied.

To continue the old, partly completed ablations instead, set `BASE_CONFIGURATION = "previous"` in cell 1. That mode loads the original validation selection and resumes the original ablation folders, including an interrupted run. Old and retuned ablation configurations are never mixed.

Run **1–5**, then any of **6–9**, then **10**. All 13 variants use three seeds, 100 epochs and the full training split. The eta grid uses the five numeric values from MOON's beta ablation. Gap is the final inner Frank–Wolfe gap on the training probe; its raw scale depends on the oracle geometry.

In [ ]:
#@title 1. Existing experiment and new Ours search
REPO_URL = "https://github.com/alirezamirrokni/LMO-MOO.git"
REPO_COMMIT = "ec912121ed084551b9898caee4334ebec3445474"
EXPERIMENT_NAME = "multimnist_a100_v2" #@param {type:"string"}
SEARCH_NAME = "ours_retune_v3" #@param {type:"string"}
DATA_CACHE_EXPERIMENT = "multimnist_a100_v1"
SEEDS = [42, 43, 44]
EPOCHS = 100
BATCH_SIZE = 256

BASE_CONFIGURATION = "retuned" #@param ["retuned", "previous"]

In [ ]:
#@title 2. Check A100, mount Drive and enable live console logs
import os, sys, json, subprocess, shutil, time, hashlib, zipfile, csv
from pathlib import Path
import torch
from google.colab import drive

assert torch.cuda.is_available(), "Select Runtime > Change runtime type > A100 GPU."
GPU_NAME = torch.cuda.get_device_name(0)
assert "A100" in GPU_NAME, f"Current GPU: {GPU_NAME}. Select A100 and reconnect."
print("GPU:", GPU_NAME, "| PyTorch:", torch.__version__)
drive.mount("/content/drive")
assert EXPERIMENT_NAME and Path(EXPERIMENT_NAME).name == EXPERIMENT_NAME and EXPERIMENT_NAME not in {".", ".."}
DRIVE_ROOT = Path("/content/drive/MyDrive/LMO-MOO")
BASE_ROOT = DRIVE_ROOT / EXPERIMENT_NAME
assert SEARCH_NAME and Path(SEARCH_NAME).name == SEARCH_NAME and SEARCH_NAME not in {".", ".."}
RUN_ROOT = BASE_ROOT / SEARCH_NAME
RUN_ROOT.mkdir(parents=True, exist_ok=True)
BASELINE_ROOT = BASE_ROOT / "results" / "multimnist"
TUNING_ROOT = RUN_ROOT / "tuning"
SELECTION_FILE = TUNING_ROOT / "selection.json"
INCUMBENT_SELECTION = BASE_ROOT / "tuning" / "selection.json"
OUTPUT_ROOT = RUN_ROOT / "results" / "multimnist"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REPO = Path("/content/LMO-MOO-retune-v3")
DATA = Path("/content/LMO-MOO-data/multimnist")
print("Persistent outputs:", OUTPUT_ROOT)


import codecs, signal, shlex, uuid
LOG_DIR = RUN_ROOT / "logs"
LOG_DIR.mkdir(exist_ok=True)

def run_live(cmd, *, cwd=None, label="process"):
    """Forward stdout/stderr chunks immediately, preserving carriage returns.

    Raw console logs are also saved to Drive. An interrupted cell stops the
    entire subprocess group, including children launched by the suite.
    """
    cmd = list(map(str, cmd))
    print("$ " + shlex.join(cmd), flush=True)
    safe_label = "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in label)
    log_path = LOG_DIR / (time.strftime("%Y%m%d_%H%M%S") + "_" + safe_label + "_" + uuid.uuid4().hex[:6] + ".log")
    env = os.environ.copy()
    env.update(PYTHONUNBUFFERED="1", PYTHONIOENCODING="utf-8")
    print("Console log:", log_path, flush=True)
    with log_path.open("wb", buffering=0) as log:
        process = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, bufsize=0,
                                   start_new_session=True)
        decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
        try:
            while True:
                chunk = os.read(process.stdout.fileno(), 8192)
                if not chunk:
                    break
                # Display first so a slow Drive write cannot hide current output.
                sys.stdout.write(decoder.decode(chunk))
                sys.stdout.flush()
                log.write(chunk)
            sys.stdout.write(decoder.decode(b"", final=True))
            sys.stdout.flush()
            returncode = process.wait()
        except BaseException:
            if process.poll() is None:
                try:
                    os.killpg(process.pid, signal.SIGTERM)
                except ProcessLookupError:
                    pass
                try:
                    process.wait(timeout=5)
                except (subprocess.TimeoutExpired, KeyboardInterrupt):
                    try:
                        os.killpg(process.pid, signal.SIGKILL)
                    except ProcessLookupError:
                        pass
                    process.wait()
            raise
        finally:
            process.stdout.close()
    print(f"\n[{label}] Exit code: {returncode}", flush=True)
    if returncode:
        raise subprocess.CalledProcessError(returncode, cmd)
    return log_path

In [ ]:
#@title 3. Load the pinned code and install dependencies
if not REPO.exists():
    run_live(["git", "clone", "--depth", "1", "--filter=blob:none", "--no-checkout", "--sparse", "--progress", REPO_URL, REPO], label="clone")
else:
    assert (REPO / ".git").exists(), f"{REPO} is not a Git checkout."
    origin = subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip()
    assert origin == REPO_URL
    dirty = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--untracked-files=no"], text=True)
    assert not dirty, "Save local tracked-file edits before rerunning setup."
if subprocess.run(["git", "-C", str(REPO), "cat-file", "-e", REPO_COMMIT + "^{commit}"], capture_output=True).returncode:
    run_live(["git", "-C", REPO, "fetch", "origin", REPO_COMMIT], label="fetch")
run_live(["git", "-C", REPO, "sparse-checkout", "set", "experiments", "methods", "scripts"], label="checkout-files")
run_live(["git", "-C", REPO, "checkout", "--detach", REPO_COMMIT], label="checkout-revision")
# Preserve Colab's installed CUDA-compatible torch and torchvision pair.
run_live([sys.executable, "-m", "pip", "install", "-r", REPO / "requirements-modern.txt", "pandas"], label="dependencies")
run_live([sys.executable, "-u", "-c", "import torch, torchvision; from experiments.multimnist.trainer import parser; print('Imports OK:', torch.__version__, torchvision.__version__)"], cwd=REPO, label="import-check")
os.chdir(REPO)
def run_script(filename, *args):
    return run_live([sys.executable, "-u", REPO / filename, *args], cwd=REPO, label=Path(filename).stem)

def atomic_json(path, value):
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(value, indent=2) + "\n")
    tmp.replace(path)

identity = {"repo": REPO_URL, "commit": REPO_COMMIT, "epochs": EPOCHS,
            "batch_size": BATCH_SIZE, "seeds": SEEDS, "dataset_seed": 2026,
            "train_samples": 10000, "test_samples": 1000,
            "baseline_root": str(BASELINE_ROOT)}
manifest = RUN_ROOT / "workflow.json"
if manifest.exists():
    assert json.loads(manifest.read_text()) == identity, "Workflow settings changed. Use a new SEARCH_NAME."
else:
    atomic_json(manifest, identity)
versions = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(RUN_ROOT / ("environment_" + time.strftime("%Y%m%d_%H%M%S") + ".txt")).write_text(versions)
print("Pinned code revision:", REPO_COMMIT)
print("Existing baselines are read from:", BASELINE_ROOT)
print("New Ours outputs are written to:", RUN_ROOT)

In [ ]:
#@title 4. Prepare and cache the fixed dataset
CACHE = DRIVE_ROOT / DATA_CACHE_EXPERIMENT / "data"
CACHE.mkdir(parents=True, exist_ok=True)
ARCHIVE = CACHE / "multimnist_seed2026.zip"
LOCAL_ARCHIVE = Path("/content/multimnist_seed2026.zip")
if not ARCHIVE.exists():
    # Remove only a partial generated dataset from an interrupted preparation.
    if DATA.exists():
        shutil.rmtree(DATA)
    run_script("prepare_multimnist.py", "--download", "--output", DATA,
               "--mnist-root", "/content/LMO-MOO-data/mnist",
               "--train-samples", 10000, "--test-samples", 1000, "--seed", 2026)
    with zipfile.ZipFile(LOCAL_ARCHIVE, "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted(DATA.rglob("*")):
            if p.is_file():
                z.write(p, p.relative_to(DATA))
    print("Saving the dataset archive to Drive...", flush=True)
    partial = ARCHIVE.with_suffix(".zip.partial")
    shutil.copy2(LOCAL_ARCHIVE, partial)
    partial.replace(ARCHIVE)
else:
    print("Restoring the dataset archive from Drive...", flush=True)
    shutil.copy2(ARCHIVE, LOCAL_ARCHIVE)

digest = hashlib.sha256(LOCAL_ARCHIVE.read_bytes()).hexdigest()
hash_file = CACHE / "dataset_archive.sha256"
if hash_file.exists():
    assert hash_file.read_text().strip() == digest, "Dataset archive checksum mismatch."
else:
    hash_file.write_text(digest + "\n")
# Reuse local data within a session; restore it after reconnecting.
marker = DATA.parent / "archive.sha256"
if not (DATA.exists() and marker.exists() and marker.read_text().strip() == digest):
    if DATA.exists():
        shutil.rmtree(DATA)
    DATA.mkdir(parents=True)
    with zipfile.ZipFile(LOCAL_ARCHIVE) as z:
        assert z.testzip() is None, "Damaged dataset archive."
        for info in z.infolist():
            target = (DATA / info.filename).resolve()
            assert target.is_relative_to(DATA.resolve()), "Unsafe archive entry."
        for info in z.infolist():
            z.extract(info, DATA)
    marker.write_text(digest + "\n")
for split, expected in [("train", 10000), ("test", 1000)]:
    with (DATA / split / "labels.csv").open() as f:
        rows = list(csv.reader(f))
    assert len(rows) == expected
    assert all((DATA / split / "2" / row[0]).is_file() for row in rows)
    print(split, len(rows), "images")
print("Dataset ready on local disk:", DATA)

In [ ]:
#@title 5. Load a frozen base configuration and inspect ablation status
from IPython.display import display
import pandas as pd
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from experiments.multimnist.trainer import parser as training_parser, run_signature, data_fingerprint
COMMON = ["--data-path", DATA, "--device", "cuda:0", "--epochs", EPOCHS,
          "--batch-size", BATCH_SIZE, "--workers", 0, "--threads", 4, "--cache-data"]
DATA_SHA256 = data_fingerprint(DATA)
TRAIN_SHA256 = data_fingerprint(DATA, splits=("train",))

def settings_flags(settings):
    keys = ["lr", "eta", "alpha", "oracle", "weights", "momentum", "lr_schedule", "min_lr_ratio"]
    return [part for k in keys if k in settings for part in ("--"+k.replace("_", "-"), settings[k])]

def load_selected(path):
    selection = json.loads(path.read_text())
    assert selection["test_used"] is False and selection["settings"]["selection"] == "validation"
    assert selection["settings"]["train_sha256"] == TRAIN_SHA256, "Selection used different images."
    selected = selection["selected"]
    assert selected["epochs"] == EPOCHS and selected["batch_size"] == BATCH_SIZE
    return selection, selected

def training_args(root, tag, method, seed, flags):
    return ["--output-root", root, "--tag", tag, "--method", method, "--seed", seed,
            *COMMON, *flags, "--resume"]

def status_for(root, tag, method, seed, flags):
    directory = root / tag / method / f"seed{seed}"
    args = training_parser().parse_args(list(map(str, training_args(root, tag, method, seed, flags))))
    expected = run_signature(args, DATA_SHA256)
    for name in ("config.json", "summary.json"):
        path = directory / name
        if path.exists() and json.loads(path.read_text()).get("signature") != expected:
            raise ValueError(f"Configuration/data mismatch: {path}")
    result = dict(tag=tag, method=method, seed=seed, status="pending", epoch=0)
    summary_path = directory / "summary.json"
    if summary_path.exists():
        s = json.loads(summary_path.read_text())
        assert s["seed"] == seed and s["method"] == method and s["tag"] == tag
        assert s.get("selection") == "test" and not s.get("smoke")
        assert 0 <= s["epoch"] <= EPOCHS
        complete = s.get("completed") is True and s["epoch"] == EPOCHS
        result.update(status="complete" if complete else "resume", epoch=s["epoch"])
        if not complete and not (directory / "checkpoint.pt").exists():
            raise FileNotFoundError(f"Partial run has no checkpoint: {directory}")
    return result

def run_ours_jobs(root, jobs):
    for tag, flags in jobs:
        for seed in SEEDS:
            status = status_for(root, tag, "ours", seed, flags)
            print(f"{tag}/ours/seed{seed}: {status['status']}", flush=True)
            if status["status"] == "complete":
                continue
            run_script("run_multimnist.py", *training_args(root, tag, "ours", seed, flags))
            assert status_for(root, tag, "ours", seed, flags)["status"] == "complete"

if BASE_CONFIGURATION == "retuned":
    selection_path = SELECTION_FILE
    ABLATION_ROOT = RUN_ROOT / "ablations" / "multimnist"
else:
    assert BASE_CONFIGURATION == "previous"
    selection_path = INCUMBENT_SELECTION
    ABLATION_ROOT = BASELINE_ROOT
assert selection_path.exists(), "Complete comparison notebook cell 8 first, or select previous mode."
selection, SELECTED = load_selected(selection_path)
BASE_FLAGS = settings_flags(SELECTED)
ABLATION_ROOT.mkdir(parents=True, exist_ok=True)
freeze = ABLATION_ROOT / "ablation_configuration.json"
if freeze.exists():
    assert json.loads(freeze.read_text()) == SELECTED, "Ablation base settings changed. Use a separate output directory."
else:
    atomic_json(freeze, SELECTED)
ABLATION_GROUPS = {}
for group, flag, values in [
    ("oracle", "--oracle", ["l2", "sign", "spectral"]),
    ("weights", "--weights", ["entropic", "projected"]),
    ("momentum", "--momentum", ["blended", "per-task", "none"]),
    ("eta", "--eta", [1e-5, 5e-5, 1e-4, 5e-4, 1e-3]),
]:
    ABLATION_GROUPS[group] = [(f"{group}-{value}", [*BASE_FLAGS, flag, value]) for value in values]

def ablation_status():
    frame = pd.DataFrame([status_for(ABLATION_ROOT, tag, "ours", seed, flags)
            for jobs in ABLATION_GROUPS.values() for tag, flags in jobs for seed in SEEDS])
    display(frame)
    frame.to_csv(ABLATION_ROOT / "ablation_status.csv", index=False)
    print(f"Complete: {(frame.status == 'complete').sum()}/{len(frame)} runs.")
    return frame
print("Base configuration:", SELECTED)
print("Output:", ABLATION_ROOT)
status = ablation_status()

In [ ]:
#@title 6. Run oracle ablations (three seeds each)
run_ours_jobs(ABLATION_ROOT, ABLATION_GROUPS["oracle"])
status = ablation_status()

In [ ]:
#@title 7. Run weights ablations (three seeds each)
run_ours_jobs(ABLATION_ROOT, ABLATION_GROUPS["weights"])
status = ablation_status()

In [ ]:
#@title 8. Run momentum ablations (three seeds each)
run_ours_jobs(ABLATION_ROOT, ABLATION_GROUPS["momentum"])
status = ablation_status()

In [ ]:
#@title 9. Run eta ablations (three seeds each)
run_ours_jobs(ABLATION_ROOT, ABLATION_GROUPS["eta"])
status = ablation_status()

In [ ]:
#@title 10. Export only the ablation table
REPORT = ABLATION_ROOT / "report_ablations"
run_script("report_multimnist.py", "--root", ABLATION_ROOT, "--section", "ablations",
           "--baseline-source", "reproduced", "--seeds", *SEEDS, "--out", REPORT)
results = pd.read_csv(REPORT / "results.csv")
display(results[["tag", "n_seeds", "avg", "avg_std", "gap", "gap_std"]])
status = ablation_status()
print("COMPLETE" if (status.status == "complete").all() else "PARTIAL: run the remaining group cells, then rerun this cell.")
atomic_json(REPORT / "base_configuration.json", SELECTED)
print("Ablation LaTeX:", REPORT / "table.tex")
print("Ablation CSV:", REPORT / "results.csv")